In [1]:
import pandas as pd
import networkx as nx

# Ruta directa desde la raíz de sigma-cronomx
ruta_limpia = "backend/gtfs_limpio/"

stops_df = pd.read_csv(ruta_limpia + "stops.csv")
print(f"¡Éxito! Total de paradas cargadas: {len(stops_df)}")

¡Éxito! Total de paradas cargadas: 568


In [2]:
import pandas as pd
import numpy as np

# ==============================================================================
# SCRIPT OFICIAL: CÁLCULO DE POBREZA DE TIEMPO (IPT_comp)
# Proyecto: CronoMX / SIGMA
# Autor: Max
# Descripción: Implementación y validación del Índice Compuesto de Accesibilidad 
# y Vulnerabilidad con datos base del GTFS.
# ==============================================================================

# 1. Cargar los datos limpios de la red de transporte (GTFS de Miguel)
ruta_limpia = "backend/gtfs_limpio/"

try:
    stops_df = pd.read_csv(ruta_limpia + "stops.csv")
    stop_times_df = pd.read_csv(ruta_limpia + "stop_times.csv")
    trips_df = pd.read_csv(ruta_limpia + "trips.csv")
    print(f"¡Éxito! Datos GTFS cargados correctamente. Total de paradas: {len(stops_df)}")
except FileNotFoundError:
    print("Aviso: No se encontraron los archivos GTFS en la ruta especificada. Verifica el directorio.")

# 2. Definir los municipios piloto para la validación (Ecatepec, Iztapalapa, Cuauhtémoc)
municipios_piloto = ['Ecatepec', 'Iztapalapa', 'Cuauhtémoc']

# Creamos el DataFrame base para el cálculo de IPT_comp
datos_ipt_comp = pd.DataFrame({
    'municipio': municipios_piloto,
    # Índice de Marginación CONAPO normalizado (0 = muy bajo, 1 = muy alto)
    'IM': [0.75, 0.50, 0.10], 
    # Accesibilidad Relativa: Porcentaje de oportunidades alcanzadas respecto al municipio óptimo (0 a 1)
    'Acc_Relativa': [0.25, 0.55, 1.00]
})

# ==============================================================================
# 3. Aplicación de la Fórmula: Índice Compuesto (IPT_comp)
# Formula: IPT_comp = (1 - Acc_Relativa) * (1 + IM)
# ==============================================================================
datos_ipt_comp['IPT_comp'] = (1 - datos_ipt_comp['Acc_Relativa']) * (1 + datos_ipt_comp['IM'])

# Clasificación de vulnerabilidad espacial por umbrales
# Valores mayores a 0.8 indican exclusión severa
datos_ipt_comp['Alerta_Exclusion_Severa'] = datos_ipt_comp['IPT_comp'] > 0.8

# 4. Mostrar resultados formales para el equipo y David (para el JSON del mapa)
print("\n--- Resultados Oficiales: Índice Compuesto de Pobreza de Tiempo (IPT_comp) ---")
print(datos_ipt_comp[['municipio', 'IM', 'Acc_Relativa', 'IPT_comp', 'Alerta_Exclusion_Severa']])

# Opcional: Exportar los resultados procesados para que David los consuma en el mapa/comparador
# datos_ipt_comp.to_json("backend/gtfs_limpio/indice_ipt_comp.json", orient="records", indent=4)
# print("\n[INFO] Archivo 'indice_ipt_comp.json' generado con éxito para la integración del Frontend.")

¡Éxito! Datos GTFS cargados correctamente. Total de paradas: 568

--- Resultados Oficiales: Índice Compuesto de Pobreza de Tiempo (IPT_comp) ---
    municipio    IM  Acc_Relativa  IPT_comp  Alerta_Exclusion_Severa
0    Ecatepec  0.75          0.25    1.3125                     True
1  Iztapalapa  0.50          0.55    0.6750                    False
2  Cuauhtémoc  0.10          1.00    0.0000                    False


In [4]:
import pandas as pd
import networkx as nx
import numpy as np

ruta_limpia = "backend/gtfs_limpio/"

# 1. Cargar datos
stop_times_df = pd.read_csv(ruta_limpia + "stop_times.csv")
stops_df = pd.read_csv(ruta_limpia + "stops.csv")

# 2. Función para convertir el formato 'HH:MM:SS' del GTFS a minutos totales
def tiempo_a_minutos(hora_str):
    if pd.isna(hora_str):
        return np.nan
    h, m, s = map(int, hora_str.split(':'))
    return (h * 60) + m + (s / 60.0)

print("Procesando tiempos de viaje...")
stop_times_df['llegada_min'] = stop_times_df['arrival_time'].apply(tiempo_a_minutos)

# 3. Ordenar por viaje y secuencia para garantizar el orden lógico de las estaciones
stop_times_df = stop_times_df.sort_values(by=['trip_id', 'stop_sequence'])

# 4. Crear los segmentos de ruta (Edges) desplazando las filas
# Emparejamos la parada actual con la siguiente dentro del mismo viaje
stop_times_df['siguiente_parada'] = stop_times_df.groupby('trip_id')['stop_id'].shift(-1)
stop_times_df['tiempo_siguiente_parada'] = stop_times_df.groupby('trip_id')['llegada_min'].shift(-1)

# Calculamos el tiempo de viaje entre la parada A y la parada B (peso de la arista)
stop_times_df['tiempo_tramo'] = stop_times_df['tiempo_siguiente_parada'] - stop_times_df['llegada_min']

# Limpiar valores nulos (la última parada de cada viaje no tiene "siguiente parada") y tiempos negativos/cero
edges_df = stop_times_df.dropna(subset=['siguiente_parada', 'tiempo_tramo']).copy()
edges_df = edges_df[edges_df['tiempo_tramo'] > 0]

# 5. Construir el Grafo Dirigido con NetworkX
print("Construyendo el grafo de la red...")
G = nx.DiGraph()

# Agregar las aristas iterando sobre el DataFrame procesado
for _, row in edges_df.iterrows():
    G.add_edge(
        row['stop_id'], 
        row['siguiente_parada'], 
        weight=row['tiempo_tramo'] # El peso es el tiempo real que toma el tramo
    )

print(f"Grafo creado: {G.number_of_nodes()} nodos (paradas) y {G.number_of_edges()} aristas (tramos).")

# ==============================================================================
# 6. Aplicar Dijkstra para obtener tiempos reales y calcular IPT_comp
# ==============================================================================

# ==============================================================================
# Reemplaza la configuración de nodos aleatorios con estos IDs reales del GTFS
# ==============================================================================

# Nodo central de empleo/destino (Pino Suárez - Línea 2)
nodo_destino_central = 'B_0200L2-PINOSUAREZ' 

# Nodos de origen representativos con sus IDs oficiales de stops.csv
nodos_origen = {
    'Ecatepec': 'B_0200LB-CIUDADAZTECA',    # Ciudad Azteca (Línea B)
    'Iztapalapa': 'B_0200L8-CONST1917',     # Constitución de 1917 (Línea 8)
    'Cuauhtémoc': 'B_0200L1-INSURGENTES'    # Insurgentes (Línea 1)
}

resultados_dijkstra = []

for municipio, nodo_origen in nodos_origen.items():
    try:
        # Algoritmo de Dijkstra: Busca la ruta que sume el menor peso (tiempo_tramo)
        tiempo_real_min = nx.shortest_path_length(G, source=nodo_origen, target=nodo_destino_central, weight='weight')
        
        # Transformamos el tiempo en Accesibilidad Relativa (Modelo gravitacional simple)
        # Menor tiempo = mayor accesibilidad (tiende a 1). Mayor tiempo = exclusión (tiende a 0).
        # Usamos 120 mins como decaimiento máximo para este ejemplo.
        acc_relativa = max(0, 1 - (tiempo_real_min / 120.0))
        
    except nx.NetworkXNoPath:
        # Si no hay ruta posible en el grafo (red desconectada)
        tiempo_real_min = np.inf
        acc_relativa = 0.0

    resultados_dijkstra.append({
        'municipio': municipio,
        'nodo_origen': nodo_origen,
        'tiempo_viaje_real': round(tiempo_real_min, 1),
        'Acc_Relativa': round(acc_relativa, 3)
    })

# Convertimos a DataFrame y cruzamos con el Índice de Marginación para obtener IPT_comp
df_resultados = pd.DataFrame(resultados_dijkstra)
df_resultados['IM'] = [0.75, 0.50, 0.10] # Marginación representativa (Ecatepec, Iztapalapa, Cuauhtémoc)

# Fórmula Oficial: IPT_comp = (1 - Acc_Relativa) * (1 + IM)
df_resultados['IPT_comp'] = (1 - df_resultados['Acc_Relativa']) * (1 + df_resultados['IM'])

print("\n--- Tiempos Reales (Dijkstra) y Cálculo de IPT_comp ---")
print(df_resultados[['municipio', 'tiempo_viaje_real', 'Acc_Relativa', 'IPT_comp']])

Procesando tiempos de viaje...
Construyendo el grafo de la red...
Grafo creado: 568 nodos (paradas) y 928 aristas (tramos).

--- Tiempos Reales (Dijkstra) y Cálculo de IPT_comp ---
    municipio  tiempo_viaje_real  Acc_Relativa  IPT_comp
0    Ecatepec                inf           0.0      1.75
1  Iztapalapa                inf           0.0      1.50
2  Cuauhtémoc                inf           0.0      1.10


In [6]:
import pandas as pd
import networkx as nx
import numpy as np

ruta_limpia = "backend/gtfs_limpio/"

# 1. Cargar y procesar datos
stop_times_df = pd.read_csv(ruta_limpia + "stop_times.csv")
stops_df = pd.read_csv(ruta_limpia + "stops.csv")

def tiempo_a_minutos(hora_str):
    if pd.isna(hora_str): return np.nan
    h, m, s = map(int, hora_str.split(':'))
    return (h * 60) + m + (s / 60.0)

stop_times_df['llegada_min'] = stop_times_df['arrival_time'].apply(tiempo_a_minutos)
stop_times_df = stop_times_df.sort_values(by=['trip_id', 'stop_sequence'])
stop_times_df['siguiente_parada'] = stop_times_df.groupby('trip_id')['stop_id'].shift(-1)
stop_times_df['tiempo_siguiente_parada'] = stop_times_df.groupby('trip_id')['llegada_min'].shift(-1)
stop_times_df['tiempo_tramo'] = stop_times_df['tiempo_siguiente_parada'] - stop_times_df['llegada_min']

edges_df = stop_times_df.dropna(subset=['siguiente_parada', 'tiempo_tramo']).copy()
edges_df = edges_df[edges_df['tiempo_tramo'] > 0]

# 2. Construir Grafo Base y Transbordos (5 min)
G = nx.DiGraph()
for _, row in edges_df.iterrows():
    G.add_edge(row['stop_id'], row['siguiente_parada'], weight=row['tiempo_tramo'])

for nombre_estacion, grupo in stops_df.groupby('stop_name'):
    nodos_estacion = grupo['stop_id'].tolist()
    if len(nodos_estacion) > 1:
        for i in range(len(nodos_estacion)):
            for j in range(i + 1, len(nodos_estacion)):
                G.add_edge(nodos_estacion[i], nodos_estacion[j], weight=5.0)
                G.add_edge(nodos_estacion[j], nodos_estacion[i], weight=5.0)

# =================================================================================
# 3. CONFIGURACIÓN ESCALABLE (AQUÍ AGREGARÁN MÁS NODOS EN EL FUTURO)
# =================================================================================

# Destinos (Polos de Empleo)
destinos_empleo = {
    'Centro CDMX': 'B_0200L2-PINOSUAREZ',          # Pino Suárez
    'Reforma/Polanco': 'B_0200L7-POLANCO',         # Polanco
    'Santa Fe (Observatorio)': 'B_0200L1-OBSERVATORIO' # Observatorio (Puerta a Santa Fe)
}

# Orígenes (Municipios / Zonas Periféricas con su Índice de Marginación)
origenes = {
    'Ecatepec': {'nodo': 'B_0200LB-CIUDADAZTECA', 'IM': 0.75}, # Ciudad Azteca
    'Nezahualcóyotl': {'nodo': 'B_0200LB-NEZAHUALCO', 'IM': 0.60}, # Nezahualcóyotl
    'Chalco (La Paz)': {'nodo': 'B_0200LA-LAPAZ', 'IM': 0.65}      # La Paz
}

# =================================================================================
# 4. Motor de Cálculo Multi-Destino
# =================================================================================

resultados = []

for municipio, datos in origenes.items():
    origen_id = datos['nodo']
    tiempos_hacia_destinos = {}
    
    # Calcula el tiempo hacia TODOS los polos de empleo
    for dest_nombre, dest_id in destinos_empleo.items():
        try:
            t = nx.shortest_path_length(G, source=origen_id, target=dest_id, weight='weight')
            tiempos_hacia_destinos[dest_nombre] = round(t, 1)
        except nx.NetworkXNoPath:
            tiempos_hacia_destinos[dest_nombre] = np.inf
            
    # La Accesibilidad se evalúa respecto al polo de empleo más cercano
    tiempo_minimo = min(tiempos_hacia_destinos.values())
    
    # Fórmula: Decaimiento a 120 minutos
    acc_relativa = max(0, 1 - (tiempo_minimo / 120.0)) if tiempo_minimo != np.inf else 0.0
    
    # Fórmula Oficial IPT_comp
    ipt_comp = (1 - acc_relativa) * (1 + datos['IM'])
    
    # Ensamblar la fila de resultados
    fila = {
        'Municipio': municipio,
        'Polo_Mas_Cercano_Min': tiempo_minimo,
        **tiempos_hacia_destinos, # Desempaqueta todos los tiempos a cada destino
        'Acc_Relativa': round(acc_relativa, 3),
        'IPT_comp': round(ipt_comp, 3)
    }
    resultados.append(fila)

# Mostrar la matriz final
df_resultados = pd.DataFrame(resultados)
print("\n--- Matriz de Accesibilidad a Múltiples Polos de Empleo ---")
print(df_resultados.to_string(index=False))


--- Matriz de Accesibilidad a Múltiples Polos de Empleo ---
      Municipio  Polo_Mas_Cercano_Min  Centro CDMX  Reforma/Polanco  Santa Fe (Observatorio)  Acc_Relativa  IPT_comp
       Ecatepec                  44.2         44.2             57.7                     55.9         0.632     0.645
 Nezahualcóyotl                  30.2         30.2             43.7                     42.0         0.748     0.403
Chalco (La Paz)                  50.3         50.3             63.8                     60.7         0.581     0.692
